### Config and setup

#### Function refresh

In [ ]:
%load_ext autoreload
%autoreload 2
# %reload_ext autoreload

In [ ]:
import sys
sys.path.insert(0, '../functions')

#### Library imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import random
from pathlib import Path
from collections import Counter

import time

import seaborn as sns
import matplotlib.pyplot as plt

from captum.attr import IntegratedGradients

torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

#### Functions import

In [ ]:
from models import DANN
from data_loading import prepare_dataloaders, load_subject_data_flexible
from metrics import evaluate

from explainability import (
    DANNClassifierWrapper, load_model_for_xai,
    fold_batchnorm, verify_folding,
    lrp_epsilon_conv2d, lrp_zplus_conv2d,
    lrp_linear, lrp_avgpool,
    compute_lrp, run_lrp_full,
    load_lrp_attributions,
    compute_per_class_accuracy_106,
    compute_deeplift, run_deeplift_full,
    load_deeplift_attributions,
    compute_shap, run_shap_full,
    load_shap_attributions,
    compute_ig, run_ig_full,
    load_ig_attributions,
)

from xai_utils import (
    assign_score_groups, get_grouped_order,
    get_score_grouped_order, classify_failure_groups,
    compute_lrp_deeplift_agreement,
    compute_method_agreement,
    compute_within_group_consistency,
    print_consistency_results,
    ch16_list
)

from xai_plots import (
    plot_group_mean_attribution,
    plot_stopping_criterion_consistency
)

#### Path

In [ ]:
base_path = Path("..")
processed_path = base_path / "Data/processed"
results_path = base_path / "results"
ch16_path = base_path / "Data/processed_16ch"

# existing attribution folders for loading LRP and DeepLIFT
lrp_attr_path = results_path / "lrp_attributions"
binary_attr_path = lrp_attr_path / "binary_cdan_16ch"
multiclass_attr_path = lrp_attr_path / "multiclass_16ch"

deeplift_attr_path = results_path / "deeplift_attributions"
binary_deeplift_path = deeplift_attr_path / "binary_cdan_16ch"
multiclass_deeplift_path = deeplift_attr_path / "multiclass_16ch"

per_class_path = results_path / "per_class_accuracy"

shap_attr_path = results_path / "shap_attributions"
binary_shap_path = shap_attr_path / "binary_cdan_16ch"
multiclass_shap_path = shap_attr_path / "multiclass_16ch"

# new IG folders
ig_attr_path = results_path / "ig_attributions"
binary_ig_zero_path = ig_attr_path / "binary_cdan_16ch_zero"
binary_ig_mean_path = ig_attr_path / "binary_cdan_16ch_mean"
multiclass_ig_zero_path = ig_attr_path / "multiclass_16ch_zero"
multiclass_ig_mean_path = ig_attr_path / "multiclass_16ch_mean"

# only creating zero baseline folders since mean is dropped
binary_ig_zero_path.mkdir(parents=True, exist_ok=True)
multiclass_ig_zero_path.mkdir(parents=True, exist_ok=True)

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

#### 16 channel importing from xai_utils

In [ ]:
# ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
#              'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

#### Valid Subjects

In [ ]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

#### Setup cell for groupings rebuild

In [ ]:
# load kappas
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    cdan_results = json.load(f)

all_kappas = {
    subj: cdan_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

# load per-class accuracy
with open(per_class_path / 'multiclass_16ch_per_class_accuracy.json') as f:
    per_class_data = json.load(f)

all_per_class = {}
for subj in valid_subjects:
    pca = per_class_data[subj]['kappa']['per_class_accuracy']
    all_per_class[subj] = {int(cls): float(acc) for cls, acc in pca.items()}

# rebuild groupings
score_groups, boundaries = assign_score_groups(all_kappas, std_multiplier=0.7)

group_a, group_b, neither = classify_failure_groups(all_per_class, all_kappas)
group_a_subjects = [s['subject'] for s in group_a]
group_b_subjects = [s['subject'] for s in group_b]

category_groups_106 = {}
for subj in valid_subjects:
    if all_kappas[subj] < 0.1:
        category_groups_106[subj] = 'negative_transfer'
    elif subj in group_a_subjects:
        category_groups_106[subj] = 'group_a_collapse'
    elif subj in group_b_subjects:
        category_groups_106[subj] = 'group_b_flat'
    else:
        category_groups_106[subj] = 'neither'

# load existing attributions for comparison
binary_attrs_106 = load_lrp_attributions(
    binary_attr_path, valid_subjects, n_classes=2)
mc_attrs_106 = load_lrp_attributions(
    multiclass_attr_path, valid_subjects, n_classes=4)
binary_dl_106 = load_deeplift_attributions(
    binary_deeplift_path, valid_subjects, n_classes=2)
mc_dl_106 = load_deeplift_attributions(
    multiclass_deeplift_path, valid_subjects, n_classes=4)

print("Setup complete")
print(f"Score groups: {dict(Counter(score_groups.values()))}")
print(f"Category groups: {dict(Counter(category_groups_106.values()))}")

### Funstions

### IG

In [ ]:
def compute_ig(
    subject,
    checkpoint_tag,
    n_classes=2,
    use_multiclass=False,
    stopping_criterion='kappa',
    baseline='zero',
    n_steps=50,
    trial_type='correct',
):
    """
    Computes Integrated Gradients attributions for one subject.

    baseline: 'zero' uses zero input as reference (default, most principled,
              satisfies completeness axiom from zero signal to actual input).
              'mean' uses mean of subject's trials (same as DeepLIFT baseline,
              useful for direct comparison with DeepLIFT results).
    n_steps: number of integration steps. 50 is standard for prototyping,
             increase to 100-200 for final results if needed.
    trial_type: 'correct', 'misclassified', or 'all'.

    Returns same format as compute_lrp, compute_deeplift, compute_shap.
    """
    import warnings

    model = load_model_for_xai(
        subject, checkpoint_tag,
        n_classes=n_classes,
        stopping_criterion=stopping_criterion
    )
    wrapper = DANNClassifierWrapper(model)
    folded = fold_batchnorm(wrapper)
    folded.eval()

    X, y = load_subject_data_flexible(
        subject, use_multiclass=use_multiclass, use_16ch=True
    )
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.LongTensor(y)

    # build baseline
    if baseline == 'mean':
        baseline_input = X_tensor.mean(dim=0, keepdim=True)
    else:
        baseline_input = torch.zeros_like(X_tensor[:1])

    ig = IntegratedGradients(folded)

    correct_attrs, correct_labels = [], []
    misc_attrs, misc_true_labels, misc_pred_labels = [], [], []

    for i in range(len(X_tensor)):
        inp = X_tensor[i:i+1]

        with torch.no_grad():
            pred = folded(inp).argmax(dim=1).item()

        true_label = y_tensor[i].item()
        is_correct = (pred == true_label)

        if trial_type == 'correct' and not is_correct:
            continue
        if trial_type == 'misclassified' and is_correct:
            continue

        attr_target = true_label if is_correct else pred

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            attr = ig.attribute(
                inp,
                baselines=baseline_input,
                target=attr_target,
                n_steps=n_steps,
            )

        attr_np = attr.detach().squeeze().numpy()

        if is_correct:
            correct_attrs.append(attr_np)
            correct_labels.append(true_label)
        else:
            misc_attrs.append(attr_np)
            misc_true_labels.append(true_label)
            misc_pred_labels.append(pred)

    def build_output(attrs, labels, n_classes):
        attributions = np.array(attrs) if attrs else np.array([])
        labels_arr = np.array(labels)
        mean_attr_per_class = {}
        for cls in range(n_classes):
            mask = labels_arr == cls
            if mask.sum() > 0:
                mean_attr_per_class[cls] = attributions[mask].mean(axis=0)
            else:
                mean_attr_per_class[cls] = None
        return attributions, labels_arr, mean_attr_per_class

    correct_out = build_output(correct_attrs, correct_labels, n_classes)
    misc_out = (
        np.array(misc_attrs) if misc_attrs else np.array([]),
        np.array(misc_true_labels),
        np.array(misc_pred_labels),
        build_output(misc_attrs, misc_pred_labels, n_classes)[2]
    )

    if trial_type == 'correct':
        return correct_out
    elif trial_type == 'misclassified':
        return misc_out
    else:
        return correct_out, misc_out

#### run_ig_full

In [ ]:
def run_ig_full(valid_subjects, checkpoint_tag, attr_save_path,
                n_classes=2, use_multiclass=False,
                baseline='zero', n_steps=50):
    """
    Runs Integrated Gradients on all valid subjects, saves per-subject attributions.
    Resume-capable: skips subjects with existing .npz files.
    Both kappa and composite criteria saved per subject.
    baseline: 'zero' (default, most principled) or 'mean'.
    n_steps: integration steps, 50 for prototyping, 100-200 for final.
    """
    attr_save_path = Path(attr_save_path)
    completed, skipped, failed = [], [], []

    for idx, subj in enumerate(valid_subjects):
        save_file = attr_save_path / f"{subj}_ig.npz"

        if save_file.exists():
            skipped.append(subj)
            continue

        try:
            save_dict = {'n_classes': n_classes}

            for criterion in ['kappa', 'composite']:
                attrs, labels, mean_by_class = compute_ig(
                    subject=subj,
                    checkpoint_tag=checkpoint_tag,
                    n_classes=n_classes,
                    use_multiclass=use_multiclass,
                    stopping_criterion=criterion,
                    baseline=baseline,
                    n_steps=n_steps,
                    trial_type='correct',
                )
                save_dict[f'labels_{criterion}'] = labels
                save_dict[f'n_trials_{criterion}'] = len(attrs)
                for cls in range(n_classes):
                    if mean_by_class[cls] is not None:
                        save_dict[f'mean_attr_class_{cls}_{criterion}'] = \
                            mean_by_class[cls]

            np.savez(save_file, **save_dict)
            completed.append(subj)

            if (idx + 1) % 10 == 0:
                print(f"Progress: {idx+1}/{len(valid_subjects)} "
                      f"({len(completed)} computed, "
                      f"{len(skipped)} skipped, "
                      f"{len(failed)} failed)")

        except Exception as e:
            print(f"FAILED {subj}: {e}")
            failed.append(subj)

    print(f"\nDone. Computed: {len(completed)}, "
          f"Skipped: {len(skipped)}, Failed: {len(failed)}")
    if failed:
        print(f"Failed: {failed}")

    return completed, skipped, failed


def load_ig_attributions(attr_save_path, valid_subjects, n_classes=2):
    """
    Loads all saved IG attribution files into memory.
    Same structure as load_lrp_attributions, load_deeplift_attributions,
    load_shap_attributions.
    """
    attr_save_path = Path(attr_save_path)
    all_results = {}
    missing = []

    for subj in valid_subjects:
        save_file = attr_save_path / f"{subj}_ig.npz"
        if not save_file.exists():
            missing.append(subj)
            continue

        data = np.load(save_file, allow_pickle=False)
        subj_results = {}

        for criterion in ['kappa', 'composite']:
            mean_by_class = {}
            for cls in range(n_classes):
                key = f'mean_attr_class_{cls}_{criterion}'
                mean_by_class[cls] = data[key] if key in data else None
            subj_results[criterion] = {
                'mean_by_class': mean_by_class,
                'n_trials': int(data[f'n_trials_{criterion}']),
            }

        all_results[subj] = subj_results

    print(f"Loaded: {len(all_results)}/{len(valid_subjects)} subjects")
    if missing:
        print(f"Missing: {missing}")

    return all_results

### End of functions

### Verifications and tests

In [ ]:
for baseline in ['zero', 'mean']:
    start = time.time()
    attrs_ig, labels_ig, mean_ig = compute_ig(
        subject='S007',
        checkpoint_tag='cdan_16ch_binary',
        n_classes=2,
        use_multiclass=False,
        stopping_criterion='kappa',
        baseline=baseline,
        n_steps=50,
        trial_type='correct',
    )
    elapsed = time.time() - start
    print(f"baseline={baseline}: {len(attrs_ig)} trials, "
          f"shape={attrs_ig[0].shape}, time={elapsed:.1f}s")
    for cls, m in mean_ig.items():
        if m is not None:
            print(f"  class {cls}: range [{m.min():.4f}, {m.max():.4f}]")

In [ ]:
lrp_map = binary_attrs_106['S007']['kappa']['mean_by_class'][0]

for baseline, mean_ig_result in [('zero', mean_ig), ]:
    # rerun mean baseline to get that result too
    pass

# run both and compare
_, _, mean_ig_zero = compute_ig('S007', 'cdan_16ch_binary',
                                  n_classes=2, baseline='zero',
                                  stopping_criterion='kappa')
_, _, mean_ig_mean = compute_ig('S007', 'cdan_16ch_binary',
                                  n_classes=2, baseline='mean',
                                  stopping_criterion='kappa')

for name, mean_result in [('zero', mean_ig_zero), ('mean', mean_ig_mean)]:
    r = np.corrcoef(lrp_map.flatten(),
                    mean_result[0].flatten())[0, 1]
    r2 = np.corrcoef(
        binary_attrs_106['S007']['kappa']['mean_by_class'][1].flatten(),
        mean_result[1].flatten())[0, 1]
    print(f"LRP vs IG ({name} baseline): left r={r:.4f}, right r={r2:.4f}")

# also compare IG mean baseline vs DeepLIFT
dl_map = binary_dl_106['S007']['kappa']['mean_by_class'][0]
r_dl = np.corrcoef(dl_map.flatten(), mean_ig_mean[0].flatten())[0, 1]
print(f"DeepLIFT vs IG (mean baseline): left r={r_dl:.4f}")

In [ ]:
check_subjects = ['S018', 'S037', 'S039']
for subj in check_subjects:
    _, _, mean_z = compute_ig(subj, 'cdan_16ch_binary',
                               n_classes=2, baseline='zero',
                               stopping_criterion='kappa')
    lrp_map = binary_attrs_106[subj]['kappa']['mean_by_class'][0]
    r = np.corrcoef(lrp_map.flatten(), mean_z[0].flatten())[0, 1]
    print(f"{subj} LRP vs IG zero: left r={r:.4f}")

### run_ig_full call for 106 subjects

In [ ]:
print("Starting IG binary CDAN-16ch, zero baseline, full 106 subjects...")
ig_bin_completed, ig_bin_skipped, ig_bin_failed = run_ig_full(
    valid_subjects=valid_subjects,
    checkpoint_tag='cdan_16ch_binary',
    attr_save_path=binary_ig_zero_path,
    n_classes=2,
    use_multiclass=False,
    baseline='zero',
    n_steps=50,
)

print("\nStarting IG multiclass-16ch, zero baseline, full 106 subjects...")
ig_mc_completed, ig_mc_skipped, ig_mc_failed = run_ig_full(
    valid_subjects=valid_subjects,
    checkpoint_tag='multiclass_16ch',
    attr_save_path=multiclass_ig_zero_path,
    n_classes=4,
    use_multiclass=True,
    baseline='zero',
    n_steps=50,
)

### Load IG results

In [ ]:
print("Loading IG binary attributions...")
binary_ig_106 = load_ig_attributions(
    binary_ig_zero_path, valid_subjects, n_classes=2
)

print("Loading IG multiclass attributions...")
mc_ig_106 = load_ig_attributions(
    multiclass_ig_zero_path, valid_subjects, n_classes=4
)

### group plots

In [ ]:
plot_group_mean_attribution(
    binary_ig_106, category_groups_106,
    score_groups, valid_subjects,
    all_kappas,
    title_tag='IG binary CDAN-16ch (zero baseline)'
)